In [18]:
!uv pip install pydantic 'pydantic[email]' --quiet

In [19]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

In [20]:
class ContactDetails(BaseModel):
    email_id: Annotated[EmailStr, Field(description='Primary email address of the patient', examples=['john@example.com'])]
    contact_number: Annotated[str, Field(min_length=10, max_length=15, description='Primary contact number', examples=['9999999999'])]
    emergency_contact_number: Annotated[Optional[str], Field(default=None, min_length=10, max_length=15, description='Emergency contact number', examples=['8888888888'])]

    @field_validator('email_id')
    @classmethod
    def email_validator(cls, value):
        valid_domain = ['domain.io', 'example.com']
        # 'example@domain.io',
        domain_name = value.split("@")[-1]

        if domain_name not in valid_domain:
            raise ValueError('Not a valid domain')
        
        return value

In [21]:
class Patient(BaseModel):

    # name: str = Field(max_length=150, description='Patient Name for records')
    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    
    age: int
    linkedin_url: Optional[AnyUrl] = None
    
    # weight: float = Field(gt=0, description='')
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    
    # married: bool = False
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    
    # allergies: Optional[List[str]] = Field(default=None, max_length=5)
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    
    contact_details: ContactDetails


    @field_validator('name')
    @classmethod
    def transform_name(cls, value):
        return value.upper()
    
    # default mode=after
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value):
        if 0 < value < 120:
            return value
        else:
            raise ValueError("Age should be in between 0 and 120")

    @model_validator(mode='after')
    def validate_emergency_contact(self):
        if self.age > 60 and self.contact_details.emergency_contact_number is None:
            raise ValueError('Emergency contact number is mandatory for patients above 60 years of age')
        return self


In [22]:
def insert_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info inserted')

In [23]:
def update_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print('Patient info updateed')

In [24]:
patient_info = {
    'name': 'Kevin',
    'age': 65,
    'weight': 67.9,
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': {
        'email_id': 'example@domain.io',
        'contact_number': '9999999999',
        'emergency_contact_number': '9999999999',
    }
}

In [25]:
patient = Patient(**patient_info)
patient

Patient(name='KEVIN', age=65, linkedin_url=None, weight=67.9, married=False, allergies=['lactose', 'dust'], contact_details=ContactDetails(email_id='example@domain.io', contact_number='9999999999', emergency_contact_number='9999999999'))

In [26]:
insert_patient_data(patient=patient)

Patient Name: KEVIN
Patient Age: 65
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number='9999999999'
Patient info inserted


In [27]:
update_patient_data(patient=patient)

Patient Name: KEVIN
Patient Age: 65
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number='9999999999'
Patient info updateed
